# Intro rapide à SQL
SQL est le langage de base de données - database (DB) - le plus répandu. Probablement à raison ! Le TP suppose des connaissances de base en SQL et utilisera un serveur MySQL à l'adresse `http://mysql`. 

SQL a connu un vif regain d'intérêt depuis l'ère cloud grâce à des service d'infra SQL managées et elastiques:
- managée : il suffit de se connecter à un cloud provider pour créer et requêter des tables
- elastique : les ressources sont adaptables à la demande, parfois rien que pour le temps d'une requête

Exemple de SQL managé : AWS Athena, Google Cloud Platform, ... Exemple chez Le Bon Coin avec AWS Athena : 200Go de tables lues, filtrées, jointure réalisées en 2 secondes.

Le SQL non managé est encore très répandu car simple à mettre en place, simple et puissant à utilliser.

In [ ]:
import os
import pymysql
import pandas as pd

# MySQL connection details
conn = pymysql.connect(
    host=os.getenv('MYSQL_HOST'),
    user=os.getenv('MYSQL_USER'),
    password=os.getenv('MYSQL_PASSWORD'),
    database=os.getenv('MYSQL_DATABASE')
)


# UC-1 : description data

- Q1: Combien y a-t-il de bières dans la DB ?
- Q2: Top10 brasseries les plus représentées avec le nombre de bière par brasserie ?
- Q3: Top10 des bières les plus fortes (ABV) en France ?
- Q4: Par pays, nombre de brasseries qui proposent des bières de type `Porter` et ABV moyen de celles-ci ?
- Q5: Mediane du nombre de bière par pays ?

In [ ]:
# your code

# UC-2 : préparer un dataset de ranking 
Tout moteur de recherche/search-engine - **SE** - nécessite de la configuration ... beaucoup de configuration. Une des configuration très orientée "data" est le calcul que l'index doit opérer pour scorer chaque réponse possible face à une requête. L'apprentissage statistique de ce score s'appelle *Learning to Rank*  - **LTR** - et nécessite des connaissances poussées en machine learning. 

Cette tâche LTR se base sur les *feedbacks implicites* des utilisateurs face au moteur de recherche. Commençons par un exemple. Quand vous cherchez un objet sur LeBonCoin, vous laissez plusieurs informations *implicites* sur votre perception des résultats proposés : les item sur lesquels vous avez cliqués bien sûr mais également ceux que vous avez probablement *vu* sans cliquer dessus ... Ces "vues sans clics" sont une précieuse information implicite sur les jugement que vous avez porté aux résultats proposés. Pour ce TP nous nous limiterons à ce concept de "vu x click" mais il est possible d'aller plus loin (dwell-time, hierarchisation des interactions explicites, ...). 

On appelle *Search Engine Results Page* - **SERP** - la liste des résultats classés par un SE. Un document qui figure dans les résulats d'une recherche a donc une position (son rang) au sein de la **SERP**.

Exemple, où :
- `query` est la recherche réalisée par un user et qui a débouché sur une SERP
- `clicked_id` : l'id de la bière cliquée par le user
- `user_id` l'id de l'utilisateur (simplifions en disant que c'est même l'id d'une recherche) : permet de retrouver tous les résultats proposés dans **une** recherche
- `id_in_serp` : l'id d'une bière figurant dans la SERP
- `pos_in_serp` : la position/le rang de la bière `id_in_serp` dans la SERP issue de la recherche 

In [ ]:
pd.read_sql_query("""
SELECT * 
FROM `beers_feedback` 
where user_id = 'ecfde3aa-7fc5-11ef-9d1e-0242ac120005'
;""", con=conn)

In [ ]:
q = "SELECT * FROM `beers_feedback` ORDER BY RAND() LIMIT 5 ;"
pd.read_sql_query(q, con=conn)

Un travail préliminaire au LTR est la constitution d'un dataset qui permet d'aggréger ces feedbacks laissés par tous les utilisateurs ayant réalisé la même query. Chacun a vu et cliqué selon ses propres impressions de pertinence et il convient de "moyenner" tout cela pour obtenir des appréciations globales. L'objectif d'un tel dataset est de pouvoir lister des exemples de triplets `(query, document, note)` qui permet de savoir que face à une *query* `milky stout low bitterness`, un *document* `Super bitter beer brewed with organic roasted barley and chocolate` aura une pertinence de *1/4* (arbitraire). 

Implémenter le modèle d'agrégation de feedback "cascade model" [1] (pour la culture, **inutile d'avoir lu l'article** pour le TD) qui propose une approche pragmatique pour obtenir ces données. La méthode est la suivante :
- pour chaque recherche utilisateur:
    - étudier la position de l'id cliqué dans la SERP - soit `clicked_pos_in_serp` cette information
    - Considérer que tout doc situés "au-dessus dans la SERP" (càd quand `pos_in_serp <= clicked_pos_in_serp`) avait été vu par l'utilisateur
    - Récapituler tous ces documents "vus et cliqués" et "vus mais pas cliqués"
- Pour chaque recherche et bière cliquée (`clicked_id`), calculer la "probabilité de clic sachant qu'elle a été vue", càd le nombre de fois qu'elle a été cliquée divisé par le nombre de fois où elle a été vue


[1] https://dl.acm.org/doi/abs/10.1145/1341531.1341545

# UC-3 : récupérer les docs qui parlent d'un mot

Peut-on utiliser SQL pour réaliser un mini moteur de recherche ? Pour différentes requêtes (`query` en anglais) textuelles très simples à base de mot-clef, retrouver les bières qui semblent répondre à la demande. Exemples :
- trouver les bières ou les brasseries qui parlent de bières "fine"
- idem pour "juicy"
- idem pour "genuine"
- idem pour les bières mâturées dans des "oak cask" (fûts en chêne) -> combien y en a-t-il ? $N_1$
   - idem pour les bières qui évoquent uniquement "cask" -> combien y en a-t-il ? $N_{1,1}$
   - idem pour celles ne parlant que de "oak" -> combien y en a-t-il ? $N_{1,2}$
- idem pour les bières qui évoquent "oak" et "cask" -> combien y en a-t-il ? $N_{2}$

# UC-4 : vectorisation des description des bières
Préparer le recours à un service de vectorisation qui permettra de convertir la connaissance sur une bière en un vecteur numérique. Ce vecteur permet de sythétiser mathématiquement l'information disponible sur une bière et sa brasserie et pourra être réutilisé plus tard dans un moteur de recherche.
à faire :
- Préparer une description la plus complète possible pour chaque bière
- envoyer ces descriptions une à une via un appel HTTP sur le vectorizer (voir ci-dessous)

## Service de vectorisation 
Un faux service de vectorisation est disponible sur le server via [http://vectorizer:5000/vectorize] + un payload JSON ayant la clef `text` qui contient le texte à vectoriser.
Exemple de véritable service de vectorization: [https://jina.ai](https://jina.ai)

In [ ]:
# your code

Essayons de construire d'avoir tous le même schéma de texte à vectoriser :
`the beer BEER_NAME from brewery BREWERY_NAME (BREWERY_DESCRIPTION) is defined as BEER_DESCRIPTION. Spec of the beer are: ABV=ABV_VALUE, IBU=IBU_VALUE, SRM=SRM_VALUE`

# UC-5 : answer question in corpa

**impossible en SQL**

**Grandes lignes :** trouvons les documents qui répondent à une question. Exemple : à partir de la description vectorisée à UC-4 pour chaque bière, comment trouver les bières qui répondent à une description plus complète ? Exemple:
- "very bitter beer with smoky taste"
- "fruity sour - balanced sourness"
- "weird beer"
